In [128]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Two libs
from initial_thinning_lib import *
from secondary_thinning_lib import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [142]:
df = pd.read_csv('data/pre-thin-data.csv')

# df = pd.read_csv('data/clean-thinning-data-2.csv')

# df = pd.read_csv('data/clean-thinning-data-3.csv')

df['status'] = df['pre_HT'].apply(lambda x: 'Alive' if pd.notnull(x) and x > 0 else 'Dead')

In [143]:
df_best3, best_start3, ranked3 = compute_best3(df)

### Removal focus on Q1/Q2

In [144]:
def _euclid2d(a_rows, a_trees, b_rows, b_trees, row_scale=1.0, tree_scale=1.0):
    dr = (a_rows[:, None] - b_rows[None, :]) * row_scale
    dt = (a_trees[:, None] - b_trees[None, :]) * tree_scale
    return np.sqrt(dr * dr + dt * dt)

# ---------------------------------------------------------------------------
# Approach-1: Q4-centric immediate-k windows, cut Q1 -> then Q2
# ---------------------------------------------------------------------------
def thin_q4_immediate_k_prefer_q12(
    df_best3,
    *,
    dbh_col='pre_DBH',
    row_col='Row',
    tree_col='Tree',
    status_col='status',
    thin_col='thin_decision',
    keep_val='Keep',
    thin_val='Thin',
    q4_fraction=0.25,          
    k_neighbors=5,              
    stand_target_removed_frac=1/3,
    target_rounding='round',    
    row_scale=1.0, tree_scale=1.0,
    verbose=False
):
    d0 = df_best3.copy()

    # Baseline = Alive & Keep after 3-row
    base_mask = d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    base_idx  = d0.index[base_mask]
    if len(base_idx) == 0:
        return d0, {'error': 'Empty baseline after 3-row'}

    base = d0.loc[base_idx]

    # Budget
    n_base = len(base)
    if target_rounding == 'floor':
        target_total = int(np.floor(n_base * stand_target_removed_frac))
    elif target_rounding == 'ceil':
        target_total = int(np.ceil(n_base * stand_target_removed_frac))
    else:
        target_total = int(np.round(n_base * stand_target_removed_frac))
    target_total = max(0, min(target_total, n_base))

    # Quartiles on baseline
    x = base[dbh_col].astype(float)
    q1_thr = float(x.quantile(0.25))
    q2_thr = float(x.quantile(0.50))   
    q3_thr = float(x.quantile(0.75))

    # Anchors: top q4_fraction by DBH
    n_anchors = max(1, int(np.ceil(n_base * q4_fraction)))
    anchors_idx = base.nlargest(n_anchors, dbh_col).index
    anchors_df  = base.loc[anchors_idx].sort_values(dbh_col, ascending=False)
    anchors_set = set(anchors_idx)

    # Candidates: Q1 & Q2 
    cand_mask = x <= q2_thr
    cand_idx  = base.index[cand_mask]

    B_rows, B_trees = base[row_col].to_numpy(float), base[tree_col].to_numpy(float)
    B_ids = np.array(list(base_idx))
    id2pos = {B_ids[i]: i for i in range(len(B_ids))}

    window = {}     
    for a in anchors_df.index:
        ai = id2pos[a]
        D = _euclid2d(np.array([B_rows[ai]]), np.array([B_trees[ai]]),
                      B_rows, B_trees, row_scale=row_scale, tree_scale=tree_scale).ravel()
        D[ai] = np.inf
        k = min(int(k_neighbors), len(B_ids) - 1)
        order = np.argpartition(D, k - 1)[:k]
        order = order[np.argsort(D[order])]  
        neigh_ids = B_ids[order]
        window[a] = neigh_ids

    d1 = d0.copy()
    budget = target_total
    picks = []

    for a in anchors_df.index:
        if budget <= 0:
            break
        neigh_ids = window[a]
        if len(neigh_ids) == 0:
            continue

        neigh_df = base.loc[neigh_ids, [dbh_col]]
        is_q1 = neigh_df[dbh_col] <= q1_thr
        is_q2 = (neigh_df[dbh_col] > q1_thr) & (neigh_df[dbh_col] <= q2_thr)

        tiered_list = list(neigh_df.index[is_q1]) + list(neigh_df.index[is_q2])
        for nid in tiered_list:
            if budget <= 0:
                break
            if nid in anchors_set:
                continue                 
            if d1.at[nid, thin_col] != keep_val:
                continue               
            d1.at[nid, thin_col] = thin_val
            picks.append(nid)
            budget -= 1

    info = {
        'baseline_count': int(n_base),
        'target_removed_total': int(target_total),
        'removed_total': int(len(pd.Index(picks).unique())),
        'unused_budget': int(budget),
        'anchors_count': int(len(anchors_idx)),
        'k_neighbors': int(k_neighbors),
        'q1_thr': q1_thr, 'q2_thr': q2_thr, 'q3_thr': q3_thr
    }
    if verbose:
        print(f"[A2] removed={len(picks)} / target={target_total}, unused={budget}, anchors={len(anchors_idx)}")

    return d1, info

# ---------------------------------------------------------------------------
# Approach-2: Weighted scoring of Q1/Q2 = (proximity to Q4) × (smallness)
# ---------------------------------------------------------------------------
def thin_q12_weighted_q4_proximity(
    df_best3,
    *,
    dbh_col='pre_DBH',
    row_col='Row',
    tree_col='Tree',
    status_col='status',
    thin_col='thin_decision',
    keep_val='Keep',
    thin_val='Thin',
    q4_fraction=0.25,          
    max_q4_per_candidate=5,    
    distance_cap=None,        
    stand_target_removed_frac=1/3,
    target_rounding='round',
    row_scale=1.0, tree_scale=1.0,
    verbose=False
):
    d0 = df_best3.copy()

    base_mask = d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    base_idx  = d0.index[base_mask]
    if len(base_idx) == 0:
        return d0, {'error': 'Empty baseline after 3-row'}

    base = d0.loc[base_idx]
    n_base = len(base)

    # Budget
    if target_rounding == 'floor':
        target_total = int(np.floor(n_base * stand_target_removed_frac))
    elif target_rounding == 'ceil':
        target_total = int(np.ceil(n_base * stand_target_removed_frac))
    else:
        target_total = int(np.round(n_base * stand_target_removed_frac))
    target_total = max(0, min(target_total, n_base))

    # Quartiles
    x = base[dbh_col].astype(float)
    q1_thr = float(x.quantile(0.25))
    q2_thr = float(x.quantile(0.50))
    q3_thr = float(x.quantile(0.75))
    xmin    = float(x.min())

    # Anchors (Q4)
    n_anchors = max(1, int(np.ceil(n_base * q4_fraction)))
    anchors_idx = base.nlargest(n_anchors, dbh_col).index
    anchors_df  = base.loc[anchors_idx].sort_values(dbh_col, ascending=False)

    # Candidates: Q1 & Q2
    cand_mask = x <= q2_thr
    cand_idx  = base.index[cand_mask]
    if len(cand_idx) == 0:
        return d0, {'baseline_count': n_base, 'target_removed_total': target_total, 'removed_total': 0}

    # Arrays
    C = base.loc[cand_idx]
    A = anchors_df

    c_rows, c_trees, c_dbh = C[row_col].to_numpy(float), C[tree_col].to_numpy(float), C[dbh_col].to_numpy(float)
    a_rows, a_trees, a_dbh = A[row_col].to_numpy(float), A[tree_col].to_numpy(float), A[dbh_col].to_numpy(float)
    cand_ids = np.array(list(cand_idx))
    # smallness in [0,1]: smaller DBH -> closer to 1
    denom = (q2_thr - xmin) if (q2_thr > xmin) else 1.0
    smallness = np.clip((q2_thr - c_dbh) / denom, 0.0, 1.0)

    D = _euclid2d(c_rows, c_trees, a_rows, a_trees, row_scale=row_scale, tree_scale=tree_scale)
    if distance_cap is not None:
        D = np.where(D <= float(distance_cap), D, np.inf)

    m = min(int(max_q4_per_candidate), D.shape[1])
    part = np.argpartition(D, m-1, axis=1)[:, :m]

    rows_idx = np.arange(D.shape[0])[:, None]
    d_small  = D[rows_idx, part]
    order_m  = np.argsort(d_small, axis=1)
    part_sorted = part[rows_idx, order_m]
    d_sorted    = d_small[rows_idx, order_m]
    a_dbh_sorted = a_dbh[part_sorted]

    # score = sum_j (anchor_dbh / (d+1)) * smallness
    with np.errstate(divide='ignore', invalid='ignore'):
        contrib = a_dbh_sorted / (d_sorted + 1.0)
    contrib[np.isinf(contrib)] = 0.0
    score = smallness * np.nansum(contrib, axis=1)

    take_n = min(target_total, len(cand_ids))
    order_global = np.argsort(-score)
    chosen = cand_ids[order_global[:take_n]]

    d1 = d0.copy()
    if len(chosen) > 0:
        d1.loc[chosen, thin_col] = thin_val

    info = {
        'baseline_count': int(n_base),
        'target_removed_total': int(target_total),
        'removed_total': int(len(chosen)),
        'unused_budget': int(target_total - len(chosen)),
        'anchors_count': int(len(anchors_idx)),
        'candidates_q12': int(len(cand_ids)),
        'params': {
            'q4_fraction': q4_fraction,
            'max_q4_per_candidate': max_q4_per_candidate,
            'distance_cap': distance_cap,
            'row_scale': row_scale,
            'tree_scale': tree_scale
        }
    }
    if verbose:
        print(f"[A4] removed={len(chosen)} / target={target_total}, unused={target_total-len(chosen)}, anchors={len(anchors_idx)}")

    return d1, info


### UI

In [145]:
pd.set_option('display.max_columns', None)   
pd.set_option('display.width', 0)            
pd.set_option('display.max_colwidth', None) 

def _run_and_show(strategy_key):
    if strategy_key == 'a2':
        label = 'Q4 immediate-k'
        df_out, info = thin_q4_immediate_k_prefer_q12(
            df_best3,
            q4_fraction=0.25,
            k_neighbors=5,
            stand_target_removed_frac=1/3,
            target_rounding='round',
            row_scale=1.0, tree_scale=1.0,
            verbose=False
        )
    elif strategy_key == 'a4':
        label = 'Q1/Q2 weighted by Q4'
        df_out, info = thin_q12_weighted_q4_proximity(
            df_best3,
            q4_fraction=0.25,
            max_q4_per_candidate=1,
            distance_cap=2, 
            stand_target_removed_frac=1/3,
            target_rounding='round',
            row_scale=1.0, tree_scale=1.0,
            verbose=False
        )
    else:
        raise ValueError("Unknown strategy key")

    tbl_vs_after = table_final_vs_after_first(
        df_after_first=df_best3, df_final=df_out,
        strategy=f'{label} vs After 3-row',
        metric='pre_DBH', vol_col='pre_stem_vol',
        status_col='status', thin_col='thin_decision'
    ).round(3).set_index('Strategy')

    display(tbl_vs_after)

    rel_tbl = anchor_release_table_immediate5(
        df_after_first=df_best3, df_final=df_out,
        treatment=label,
        top_pct_anchors=0.25,  # Q4 by definition
        neighbors_k=5,
        dbh_col='pre_DBH', row_col='Row', tree_col='Tree',
        status_col='status', thin_col='thin_decision',
        keep_val='Keep', thin_val='Thin',
        row_scale=1.0, tree_scale=1.0
    ).round(3).set_index('Treatment')
    display(rel_tbl)

    # Spatial map
    plot_thinning_map(df_out, title=label)

dd = widgets.Dropdown(
    options=[
        ('Q4 immediate-k', 'a2'),
        ('Q1/Q2 weighted by Q4', 'a4'),
    ],
    value='a2',
    description='Strategy:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='520px')
)

out = widgets.Output()

def _on_change(change):
    if change['name'] == 'value' and change['type'] == 'change':
        with out:
            clear_output(wait=True)
            _run_and_show(change['new'])

dd.observe(_on_change, names='value')
display(dd, out)

with out:
    clear_output(wait=True)
    _run_and_show(dd.value)


Dropdown(description='Strategy:', layout=Layout(width='520px'), options=(('Q4 immediate-k', 'a2'), ('Q1/Q2 wei…

Output()